## Cell 1 — Imports & Configuration

**HPC variant** — runs on the Sunway HPC (AWS Linux, Open OnDemand).  
Working directory: `/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2_24020059`  
GPU (`cuda`) is assumed available; TotalSegmentator runs the full model (`fast=False`).

In [ ]:
%matplotlib inline
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
from time import time
import tempfile
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from totalsegmentator.python_api import totalsegmentator

# ============================================================
# Configuration — HPC paths (Linux)
# ============================================================
PROJECT_ROOT = Path("/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2_24020059")

VSD_RAW_DIR   = PROJECT_ROOT / "data" / "raw" / "VSD_Dataset"
MERGED_VSD_DIR = PROJECT_ROOT / "data" / "interim" / "merged_vsd"
OUTPUT_DIR    = PROJECT_ROOT / "data" / "raw" / "healthy"
MASK_DIR      = PROJECT_ROOT / "data" / "interim" / "bone_masks"
MASK_DIR.mkdir(parents=True, exist_ok=True)

# Cropping parameters (matching existing pipeline)
KNEE_CROP_MARGIN_MM = 100.0
XY_PAD_PIXELS       = 15
BONE_HU_THRESHOLD   = 200
EDGE_MARGIN_PX      = 3

# TotalSegmentator knee bone labels
# total task: femur_left (75), femur_right (76)
# appendicular_bones task: patella (1), tibia (2), fibula (3)
FEMUR_LABELS        = {"femur_left": 75, "femur_right": 76}
APPENDICULAR_LABELS = {"patella": 1, "tibia": 2, "fibula": 3}

# Test subjects: 2 z-prefix + 2 merged
Z_PREFIX_SUBJECTS = ["z001", "z004"]
MERGED_SUBJECTS   = ["016", "017"]

# Device: GPU on HPC
DEVICE = "cuda"

# Full model on GPU (fast=False gives better segmentation quality)
TS_FAST = False

print(f"Project root: {PROJECT_ROOT}")
print(f"Mask output:  {MASK_DIR}")
print(f"Device:       {DEVICE}")
print(f"TotalSegmentator fast mode: {TS_FAST}")
print(f"Test subjects: z-prefix={Z_PREFIX_SUBJECTS}, merged={MERGED_SUBJECTS}")

## Cell 2 — Data Discovery & Loading Helpers

Discover test subject volumes and define helpers to load DICOM series or NIfTI files
into a consistent format for TotalSegmentator.

In [ ]:
def load_dicom_as_nifti_path(dicom_dir):
    """Load a DICOM series and save as temporary NIfTI for TotalSegmentator."""
    series_dirs = [d for d in dicom_dir.iterdir() if d.is_dir()]
    if not series_dirs:
        raise FileNotFoundError(f"No series directories in {dicom_dir}")

    series_dir = series_dirs[0]
    reader = sitk.ImageSeriesReader()
    dicom_files = reader.GetGDCMSeriesFileNames(str(series_dir))
    if not dicom_files:
        raise FileNotFoundError(f"No DICOM files in {series_dir}")

    reader.SetFileNames(dicom_files)
    reader.MetaDataDictionaryArrayUpdateOn()
    reader.LoadPrivateTagsOn()
    img = reader.Execute()

    tmp = tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False)
    sitk.WriteImage(img, tmp.name)
    return tmp.name, img


def get_volume_info(img):
    """Extract key metadata from a SimpleITK image."""
    size    = img.GetSize()
    spacing = img.GetSpacing()
    return {
        "size_xyz":       size,
        "spacing_mm":     tuple(round(s, 4) for s in spacing),
        "phys_extent_mm": tuple(round(s * sp, 1) for s, sp in zip(size, spacing)),
    }


# ── Discover test subjects ──────────────────────────────────────────────────
subjects = []

for sid in Z_PREFIX_SUBJECTS:
    dicom_dir = VSD_RAW_DIR / sid
    if dicom_dir.exists():
        subjects.append({"id": sid, "type": "z-prefix", "path": dicom_dir, "format": "dicom"})
    else:
        print(f"  [SKIP] {sid}: DICOM dir not found at {dicom_dir}")

for sid in MERGED_SUBJECTS:
    nifti_path = MERGED_VSD_DIR / f"VSD_{sid}_merged.nii.gz"
    if nifti_path.exists():
        subjects.append({"id": sid, "type": "merged", "path": nifti_path, "format": "nifti"})
    else:
        print(f"  [SKIP] {sid}: NIfTI not found at {nifti_path}")

print(f"Found {len(subjects)} test subjects:")
for s in subjects:
    print(f"  {s['id']} ({s['type']}, {s['format']}): {s['path']}")

## Cell 3 — TotalSegmentator Test on Single Volume

Run both `total` (femur L/R) and `appendicular_bones` (tibia, patella, fibula) tasks
on one volume to validate the approach.

**HPC note**: `fast=False` — full model runs on GPU. Expected runtime: ~1-3 min per task.

In [ ]:
# ── Test on one merged subject (smaller than full-body, faster) ──────────────
test_subject = subjects[-1]  # last merged subject
sid = test_subject["id"]
print(f"Test subject: {sid} ({test_subject['type']})")

# Load volume
if test_subject["format"] == "nifti":
    input_path = str(test_subject["path"])
    img_sitk   = sitk.ReadImage(input_path)
else:
    input_path, img_sitk = load_dicom_as_nifti_path(test_subject["path"])

info = get_volume_info(img_sitk)
print(f"  Size: {info['size_xyz']}")
print(f"  Spacing: {info['spacing_mm']} mm")
print(f"  Physical extent: {info['phys_extent_mm']} mm")

nib_img = nib.load(input_path)
print(f"  NIfTI shape (nibabel): {nib_img.shape}")
print(f"  NIfTI affine diagonal (spacing): {np.abs(np.diag(nib_img.affine)[:3]).round(3)}")

# ── Run TotalSegmentator: total task (femur L/R) ────────────────────────────
print(f"\nRunning 'total' task (femur identification, fast={TS_FAST}, device={DEVICE})...")
t0 = time()
seg_total = totalsegmentator(
    input_path,
    None,
    fast=TS_FAST,
    device=DEVICE,
    quiet=True,
)
t_total = time() - t0
print(f"  Done in {t_total:.1f}s")

seg_total_arr = seg_total.get_fdata().astype(np.int16)
unique_labels = np.unique(seg_total_arr)
print(f"  Unique labels found: {len(unique_labels)} (including background)")

# Check for femur labels
from totalsegmentator.map_to_binary import class_map
total_map = class_map["total"]
for label_id in unique_labels:
    if label_id == 0:
        continue
    name = total_map.get(int(label_id), f"unknown_{label_id}")
    if "femur" in name or "hip" in name:
        n_voxels = (seg_total_arr == label_id).sum()
        print(f"  Label {label_id} ({name}): {n_voxels:,} voxels")

# ── Run TotalSegmentator: appendicular_bones task ───────────────────────────
print(f"\nRunning 'appendicular_bones' task (tibia, patella, fibula, fast={TS_FAST})...")
t0 = time()
seg_appendicular = totalsegmentator(
    input_path,
    None,
    fast=TS_FAST,
    device=DEVICE,
    task="appendicular_bones",
    quiet=True,
)
t_append = time() - t0
print(f"  Done in {t_append:.1f}s")

seg_append_arr = seg_appendicular.get_fdata().astype(np.int16)
append_map = class_map["appendicular_bones"]
for label_id in np.unique(seg_append_arr):
    if label_id == 0:
        continue
    name      = append_map.get(int(label_id), f"unknown_{label_id}")
    n_voxels  = (seg_append_arr == label_id).sum()
    print(f"  Label {label_id} ({name}): {n_voxels:,} voxels")

print(f"\nTotal runtime: {t_total + t_append:.1f}s")

## Cell 4 — Visualize Segmentation on Test Volume

3-plane view showing the raw CT volume overlaid with bone segmentation labels.
Confirms TotalSegmentator correctly identifies knee bones in the volume.

In [ ]:
# ── Build combined knee bone mask ─────────────────────────────────────────────
femur_left  = (seg_total_arr == FEMUR_LABELS["femur_left"])
femur_right = (seg_total_arr == FEMUR_LABELS["femur_right"])
tibia       = (seg_append_arr == APPENDICULAR_LABELS["tibia"])
patella     = (seg_append_arr == APPENDICULAR_LABELS["patella"])
fibula      = (seg_append_arr == APPENDICULAR_LABELS["fibula"])
all_bones   = femur_left | femur_right | tibia | patella | fibula

print(f"Femur left:  {femur_left.sum():>10,} voxels")
print(f"Femur right: {femur_right.sum():>10,} voxels")
print(f"Tibia:       {tibia.sum():>10,} voxels")
print(f"Patella:     {patella.sum():>10,} voxels")
print(f"Fibula:      {fibula.sum():>10,} voxels")
print(f"All bones:   {all_bones.sum():>10,} voxels")

raw_arr = nib_img.get_fdata()

label_overlay = np.zeros_like(seg_total_arr, dtype=np.uint8)
label_overlay[femur_left]  = 1
label_overlay[femur_right] = 2
label_overlay[tibia]       = 3
label_overlay[patella]     = 4
label_overlay[fibula]      = 5

bone_z_coords = np.where(all_bones.any(axis=(0, 1)))[0]
knee_z_mid = bone_z_coords[len(bone_z_coords) // 2] if len(bone_z_coords) > 0 else raw_arr.shape[2] // 2
bone_y_coords = np.where(all_bones.any(axis=(0, 2)))[0]
bone_x_coords = np.where(all_bones.any(axis=(1, 2)))[0]
mid_y = bone_y_coords[len(bone_y_coords) // 2] if len(bone_y_coords) > 0 else raw_arr.shape[1] // 2
mid_x = bone_x_coords[len(bone_x_coords) // 2] if len(bone_x_coords) > 0 else raw_arr.shape[0] // 2

bone_cmap = ListedColormap(["none", "red", "blue", "green", "yellow", "orange"])
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# Axial
ax_slice  = raw_arr[:, :, knee_z_mid]
lab_slice = label_overlay[:, :, knee_z_mid]
axes[0].imshow(ax_slice.T, cmap="gray", vmin=-500, vmax=1500, origin="lower")
overlay = np.ma.masked_where(~(lab_slice > 0).T, lab_slice.T)
axes[0].imshow(overlay, cmap=bone_cmap, vmin=0, vmax=5, alpha=0.5, origin="lower")
axes[0].set_title(f"Axial (z={knee_z_mid})")
axes[0].axis("off")

# Coronal
cor_slice = raw_arr[:, mid_y, :]
lab_cor   = label_overlay[:, mid_y, :]
axes[1].imshow(cor_slice.T, cmap="gray", vmin=-500, vmax=1500, origin="lower", aspect="auto")
overlay = np.ma.masked_where(~(lab_cor > 0).T, lab_cor.T)
axes[1].imshow(overlay, cmap=bone_cmap, vmin=0, vmax=5, alpha=0.5, origin="lower", aspect="auto")
axes[1].set_title(f"Coronal (y={mid_y})")
axes[1].axis("off")

# Sagittal
sag_slice = raw_arr[mid_x, :, :]
lab_sag   = label_overlay[mid_x, :, :]
axes[2].imshow(sag_slice.T, cmap="gray", vmin=-500, vmax=1500, origin="lower", aspect="auto")
overlay = np.ma.masked_where(~(lab_sag > 0).T, lab_sag.T)
axes[2].imshow(overlay, cmap=bone_cmap, vmin=0, vmax=5, alpha=0.5, origin="lower", aspect="auto")
axes[2].set_title(f"Sagittal (x={mid_x})")
axes[2].axis("off")

plt.suptitle(
    f"VSD {sid} — Bone Segmentation "
    "(red=femur_L, blue=femur_R, green=tibia, yellow=patella, orange=fibula)",
    fontsize=12,
)
plt.tight_layout()
plt.show()

print(f"\nKnee region z-range: {bone_z_coords.min()} – {bone_z_coords.max()} "
      f"(center: {knee_z_mid})")
del raw_arr

## Cell 5 — Knee Region Extraction Functions

Core logic: use segmentation labels to find knee center, separate bilateral knees,
crop, and create bone-only masks. Replaces the heuristic bone-profile pipeline.

In [ ]:
def find_knee_centers_from_segmentation(seg_total_arr, seg_append_arr, spacing):
    """
    Find bilateral knee centers using TotalSegmentator labels.

    Uses the distal femur / proximal tibia gap to locate each knee joint.
    The femur labels (total task) provide left/right separation; tibia voxels
    (appendicular_bones) are assigned to sides by x-proximity to femur centroids.

    Parameters
    ----------
    seg_total_arr  : ndarray (x, y, z) — nibabel axis ordering
    seg_append_arr : ndarray (x, y, z)
    spacing        : tuple (sx, sy, sz) in mm

    Returns
    -------
    list of dict with keys: side, knee_center_z, femur_z_range,
                            tibia_z_range, bbox_x, bbox_y, bone_mask, ...
    """
    femur_L     = (seg_total_arr  == FEMUR_LABELS["femur_left"])
    femur_R     = (seg_total_arr  == FEMUR_LABELS["femur_right"])
    tibia_all   = (seg_append_arr == APPENDICULAR_LABELS["tibia"])
    patella_all = (seg_append_arr == APPENDICULAR_LABELS["patella"])
    fibula_all  = (seg_append_arr == APPENDICULAR_LABELS["fibula"])

    knees = []
    for side, femur_mask in [("Left", femur_L), ("Right", femur_R)]:
        if femur_mask.sum() == 0:
            print(f"  WARNING: No femur_{side.lower()} voxels found, skipping")
            continue

        femur_x_coords = np.where(femur_mask.any(axis=(1, 2)))[0]
        femur_x_center = femur_x_coords.mean()
        femur_z_coords = np.where(femur_mask.any(axis=(0, 1)))[0]
        femur_z_min, femur_z_max = femur_z_coords.min(), femur_z_coords.max()

        # Assign appendicular bones to this side by x-midpoint between femurs
        other_femur = femur_R if side == "Left" else femur_L
        if other_femur.sum() > 0:
            other_x    = np.where(other_femur.any(axis=(1, 2)))[0].mean()
            x_midpoint = int((femur_x_center + other_x) / 2)
        else:
            x_midpoint = int(femur_x_center)

        def split_side(mask):
            m = mask.copy()
            if side == "Left":
                m[x_midpoint:, :, :] = False
            else:
                m[:x_midpoint, :, :] = False
            return m

        tibia_side   = split_side(tibia_all)
        patella_side = split_side(patella_all)
        fibula_side  = split_side(fibula_all)

        tibia_z_coords = np.where(tibia_side.any(axis=(0, 1)))[0]

        # Knee joint center: midpoint between distal femur and proximal tibia
        if len(tibia_z_coords) > 0:
            tibia_z_min, tibia_z_max = tibia_z_coords.min(), tibia_z_coords.max()
            knee_z = (femur_z_min + tibia_z_max) // 2
        else:
            tibia_z_min = tibia_z_max = None
            knee_z = femur_z_min

        side_bones = femur_mask | tibia_side | patella_side | fibula_side
        bone_x = np.where(side_bones.any(axis=(1, 2)))[0]
        bone_y = np.where(side_bones.any(axis=(0, 2)))[0]

        knees.append({
            "side":            side,
            "knee_center_z":   int(knee_z),
            "knee_center_z_mm": round(knee_z * spacing[2], 1),
            "femur_z_range":   (int(femur_z_min), int(femur_z_max)),
            "tibia_z_range":   (int(tibia_z_min), int(tibia_z_max)) if tibia_z_min is not None else None,
            "bbox_x":          (int(bone_x.min()), int(bone_x.max()) + 1) if len(bone_x) > 0 else None,
            "bbox_y":          (int(bone_y.min()), int(bone_y.max()) + 1) if len(bone_y) > 0 else None,
            "femur_mask":      femur_mask,
            "tibia_mask":      tibia_side,
            "patella_mask":    patella_side,
            "fibula_mask":     fibula_side,
            "bone_mask":       side_bones,
        })

    return knees


def crop_knee_from_segmentation(nib_img, knee_info,
                                crop_margin_mm=KNEE_CROP_MARGIN_MM,
                                xy_pad=XY_PAD_PIXELS):
    """
    Crop a single knee volume using segmentation-derived bounds.

    Returns cropped volume (SimpleITK image), bone mask, and crop metadata.
    """
    arr     = nib_img.get_fdata()
    affine  = nib_img.affine
    spacing = np.abs(np.diag(affine)[:3])

    knee_z       = knee_info["knee_center_z"]
    margin_slices = int(crop_margin_mm / spacing[2])

    z_min = max(0, knee_z - margin_slices)
    z_max = min(arr.shape[2], knee_z + margin_slices)

    bx    = knee_info["bbox_x"]
    by    = knee_info["bbox_y"]
    x_min = max(0, bx[0] - xy_pad)
    x_max = min(arr.shape[0], bx[1] + xy_pad)
    y_min = max(0, by[0] - xy_pad)
    y_max = min(arr.shape[1], by[1] + xy_pad)

    cropped_arr  = arr[x_min:x_max, y_min:y_max, z_min:z_max].copy()
    cropped_mask = knee_info["bone_mask"][x_min:x_max, y_min:y_max, z_min:z_max].copy()

    # nibabel (x,y,z) → SimpleITK array (z,y,x)
    sitk_arr      = np.transpose(cropped_arr, (2, 1, 0)).astype(np.float32)
    sitk_mask_arr = np.transpose(cropped_mask.astype(np.uint8), (2, 1, 0))

    cropped_img = sitk.GetImageFromArray(sitk_arr)
    cropped_img.SetSpacing((float(spacing[0]), float(spacing[1]), float(spacing[2])))
    mask_img = sitk.GetImageFromArray(sitk_mask_arr)
    mask_img.SetSpacing((float(spacing[0]), float(spacing[1]), float(spacing[2])))

    new_origin = affine @ np.array([x_min, y_min, z_min, 1.0])
    cropped_img.SetOrigin(new_origin[:3].tolist())
    mask_img.SetOrigin(new_origin[:3].tolist())

    crop_meta = {
        "z_min": z_min, "z_max": z_max,
        "y_min": y_min, "y_max": y_max,
        "x_min": x_min, "x_max": x_max,
        "cropped_shape_zyx": sitk_arr.shape,
        "phys_z_mm":  round((z_max - z_min) * spacing[2], 1),
        "phys_x_mm":  round((x_max - x_min) * spacing[0], 1),
        "phys_y_mm":  round((y_max - y_min) * spacing[1], 1),
        "bone_voxels":  int(cropped_mask.sum()),
        "total_voxels": int(np.prod(cropped_arr.shape)),
        "bone_ratio":   round(cropped_mask.sum() / np.prod(cropped_arr.shape), 4),
    }

    return cropped_img, mask_img, crop_meta


print("Knee extraction functions defined.")

## Cell 6 — Extract & Visualize Knees from Test Volume

Apply the extraction functions to the test volume from Cell 3. Shows the cropped
knee volumes and bone masks for both left and right knees.

In [ ]:
# ── Find knee centers from segmentation ──────────────────────────────────────
spacing = np.abs(np.diag(nib_img.affine)[:3])
knees   = find_knee_centers_from_segmentation(seg_total_arr, seg_append_arr, spacing)

print(f"Found {len(knees)} knees:\n")
for k in knees:
    print(f"  {k['side']}:")
    print(f"    Knee center: z={k['knee_center_z']} ({k['knee_center_z_mm']} mm)")
    print(f"    Femur z-range: {k['femur_z_range']}")
    print(f"    Tibia z-range: {k['tibia_z_range']}")
    print(f"    XY bbox: x={k['bbox_x']}, y={k['bbox_y']}")

# ── Crop both knees ──────────────────────────────────────────────────────────
print("\nCropping knees...")
test_crops = []
for knee in knees:
    cropped_img, mask_img, meta = crop_knee_from_segmentation(nib_img, knee)
    test_crops.append({
        "side":        knee["side"],
        "cropped_img": cropped_img,
        "mask_img":    mask_img,
        "meta":        meta,
    })
    print(f"  {knee['side']}: shape={meta['cropped_shape_zyx']}, "
          f"bone_ratio={meta['bone_ratio']:.3f}, "
          f"phys=({meta['phys_x_mm']}x{meta['phys_y_mm']}x{meta['phys_z_mm']}) mm")

# ── Visualize cropped knees with bone mask overlay ───────────────────────────
for crop in test_crops:
    arr  = sitk.GetArrayFromImage(crop["cropped_img"])  # (z, y, x)
    mask = sitk.GetArrayFromImage(crop["mask_img"])
    mid  = [s // 2 for s in arr.shape]

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))

    # Row 1: Raw CT
    axes[0, 0].imshow(arr[mid[0]],         cmap="gray", vmin=-500, vmax=1500)
    axes[0, 0].set_title(f"Axial (z={mid[0]})")
    axes[0, 0].axis("off")
    axes[0, 1].imshow(arr[:, mid[1], :],   cmap="gray", vmin=-500, vmax=1500, aspect="auto")
    axes[0, 1].set_title("Coronal")
    axes[0, 1].axis("off")
    axes[0, 2].imshow(arr[:, :, mid[2]],   cmap="gray", vmin=-500, vmax=1500, aspect="auto")
    axes[0, 2].set_title("Sagittal")
    axes[0, 2].axis("off")

    # Row 2: CT + bone mask overlay
    for col, (sl, lb) in enumerate([
        (arr[mid[0]],       mask[mid[0]]),
        (arr[:, mid[1], :], mask[:, mid[1], :]),
        (arr[:, :, mid[2]], mask[:, :, mid[2]]),
    ]):
        asp = "auto" if col > 0 else None
        axes[1, col].imshow(sl, cmap="gray", vmin=-500, vmax=1500, aspect=asp)
        red = np.zeros((*sl.shape, 4))
        red[lb > 0] = [1, 0, 0, 0.4]
        axes[1, col].imshow(red, aspect=asp)
        axes[1, col].axis("off")

    axes[0, 0].set_ylabel("Raw CT", fontsize=12)
    axes[1, 0].set_ylabel("CT + Bone Mask", fontsize=12)

    plt.suptitle(
        f"VSD {sid} {crop['side']} — Segmentation-Based Knee Crop "
        f"(bone ratio: {crop['meta']['bone_ratio']:.3f})",
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()

    del arr, mask

## Cell 7 — Batch Processing (All Test Subjects)

Process all 4 test subjects (2 z-prefix + 2 merged). For each:
1. Load volume (DICOM → NIfTI conversion for z-prefix)
2. Run TotalSegmentator (`total` + `appendicular_bones`)
3. Find knee centers, crop bilateral knees
4. Save cropped volumes + bone masks

**HPC note**: Full model (`fast=False`) on GPU. Expected runtime: ~5-15 min total.

In [ ]:
import os

all_metadata = []
saved_count  = 0

for subj in subjects:
    sid = subj["id"]
    print(f"\n{'='*60}")
    print(f"Processing {sid} ({subj['type']})...")
    print(f"{'='*60}")

    # ── Load volume ──────────────────────────────────────────────────────
    t0 = time()
    if subj["format"] == "nifti":
        input_path   = str(subj["path"])
        img_sitk     = sitk.ReadImage(input_path)
    else:
        input_path, img_sitk = load_dicom_as_nifti_path(subj["path"])

    nib_img_batch = nib.load(input_path)
    info          = get_volume_info(img_sitk)
    print(f"  Loaded: {info['size_xyz']}, spacing={info['spacing_mm']} mm ({time()-t0:.1f}s)")

    # ── Run TotalSegmentator ─────────────────────────────────────────────
    print(f"  Running 'total' task (fast={TS_FAST}, device={DEVICE})...")
    t0 = time()
    seg_total_batch = totalsegmentator(
        input_path, None, fast=TS_FAST, device=DEVICE, quiet=True
    )
    t_total = time() - t0
    print(f"    Done ({t_total:.1f}s)")

    print(f"  Running 'appendicular_bones' task (fast={TS_FAST})...")
    t0 = time()
    seg_append_batch = totalsegmentator(
        input_path, None, fast=TS_FAST, device=DEVICE,
        task="appendicular_bones", quiet=True
    )
    t_append = time() - t0
    print(f"    Done ({t_append:.1f}s)")

    seg_total_arr_batch  = seg_total_batch.get_fdata().astype(np.int16)
    seg_append_arr_batch = seg_append_batch.get_fdata().astype(np.int16)

    # ── Find knees and crop ──────────────────────────────────────────────
    spacing_batch = np.abs(np.diag(nib_img_batch.affine)[:3])
    knees_batch   = find_knee_centers_from_segmentation(
        seg_total_arr_batch, seg_append_arr_batch, spacing_batch
    )
    print(f"  Found {len(knees_batch)} knees")

    for knee in knees_batch:
        side = knee["side"]
        cropped_img, mask_img, meta = crop_knee_from_segmentation(nib_img_batch, knee)

        # Save cropped volume
        subject_dir = OUTPUT_DIR / f"VSD.{sid}"
        subject_dir.mkdir(parents=True, exist_ok=True)
        vol_path = subject_dir / f"VSD_{sid}_{side}.nii.gz"
        sitk.WriteImage(cropped_img, str(vol_path))

        # Save bone mask
        mask_subdir = MASK_DIR / f"VSD.{sid}"
        mask_subdir.mkdir(parents=True, exist_ok=True)
        mask_path = mask_subdir / f"VSD_{sid}_{side}_bone.nii.gz"
        sitk.WriteImage(mask_img, str(mask_path))

        meta.update({
            "subject_id":      sid,
            "side":            side,
            "source_type":     subj["type"],
            "knee_center_z":   knee["knee_center_z"],
            "knee_center_z_mm": knee["knee_center_z_mm"],
            "femur_z_range":   str(knee["femur_z_range"]),
            "tibia_z_range":   str(knee["tibia_z_range"]),
            "output_vol":      str(vol_path),
            "output_mask":     str(mask_path),
            "seg_runtime_s":   round(t_total + t_append, 1),
            "ts_fast":         TS_FAST,
            "device":          DEVICE,
        })
        all_metadata.append(meta)
        saved_count += 1

        print(f"    {side}: shape={meta['cropped_shape_zyx']}, "
              f"bone_ratio={meta['bone_ratio']:.3f}")

    # Clean up temporary NIfTI for DICOM subjects
    if subj["format"] == "dicom":
        os.unlink(input_path)

    del nib_img_batch, seg_total_batch, seg_append_batch
    del seg_total_arr_batch, seg_append_arr_batch

print(f"\n{'='*60}")
print(f"Batch complete: {saved_count} knee volumes + bone masks saved.")
print(f"{'='*60}")

## Cell 8 — Validation

For all processed volumes:
1. Boundary contact check (bone within 3px of crop edge → truncation)
2. Bone-only masked volume demo (windowed volume × bone mask)
3. 3-plane visualization of all crops

In [ ]:
def check_boundary_contact(fpath, edge_px=EDGE_MARGIN_PX, bone_hu=BONE_HU_THRESHOLD):
    """Check if bone voxels touch within edge_px pixels of any XY face."""
    arr  = sitk.GetArrayFromImage(sitk.ReadImage(str(fpath)))  # (z, y, x)
    bone = arr > bone_hu
    contact = {
        "x_min": bool(bone[:, :, :edge_px].any()),
        "x_max": bool(bone[:, :, -edge_px:].any()),
        "y_min": bool(bone[:, :edge_px, :].any()),
        "y_max": bool(bone[:, -edge_px:, :].any()),
    }
    contact["truncated"] = any(contact.values())
    return contact


# ── Boundary contact check ───────────────────────────────────────────────────
print("Boundary contact analysis:\n")
all_pass = True
for meta in all_metadata:
    sid     = meta["subject_id"]
    side    = meta["side"]
    contact = check_boundary_contact(meta["output_vol"])
    status  = "PASS" if not contact["truncated"] else "FAIL"
    if contact["truncated"]:
        faces = [k for k, v in contact.items() if v and k != "truncated"]
        print(f"  [{status}] {sid} {side}: bone at {', '.join(faces)}")
        all_pass = False
    else:
        print(f"  [{status}] {sid} {side}")

print(f"\n{'All volumes pass.' if all_pass else 'Some volumes have boundary contact!'}")

# ── Bone-only masking demo ───────────────────────────────────────────────────
HU_MIN, HU_MAX = -450, 1050
print(f"\n--- Bone-Only Masking Demo (HU window [{HU_MIN}, {HU_MAX}]) ---\n")

n = len(all_metadata)
fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for i, meta in enumerate(all_metadata):
    vol_arr  = sitk.GetArrayFromImage(sitk.ReadImage(meta["output_vol"]))
    mask_arr = sitk.GetArrayFromImage(sitk.ReadImage(meta["output_mask"]))
    mid      = [s // 2 for s in vol_arr.shape]

    windowed = np.clip((vol_arr - HU_MIN) / (HU_MAX - HU_MIN), 0, 1)
    masked   = windowed * mask_arr

    sid  = meta["subject_id"]
    side = meta["side"]

    for col, (title, sl_w, sl_m) in enumerate([
        ("Axial",    windowed[mid[0]],       masked[mid[0]]),
        ("Coronal",  windowed[:, mid[1], :], masked[:, mid[1], :]),
        ("Sagittal", windowed[:, :, mid[2]], masked[:, :, mid[2]]),
    ]):
        asp      = "auto" if col > 0 else None
        combined = sl_w * 0.2 + sl_m * 0.8
        axes[i, col].imshow(combined, cmap="gray", vmin=0, vmax=1, aspect=asp)
        axes[i, col].set_title(title if i == 0 else "")
        axes[i, col].axis("off")

    axes[i, 0].set_ylabel(f"{sid} {side}\n(bone: {meta['bone_ratio']:.3f})", fontsize=10)
    del vol_arr, mask_arr

plt.suptitle("Bone-Only Masked Volumes (bright=bone, dim=context)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Cell 9 — Metadata Export & Summary

In [ ]:
import pandas as pd

df_meta = pd.DataFrame(all_metadata)

meta_path = MASK_DIR / "bone_segmentation_metadata.csv"
df_meta.to_csv(meta_path, index=False)
print(f"Metadata saved: {meta_path}\n")

display_cols = [
    "subject_id", "side", "source_type", "cropped_shape_zyx",
    "phys_x_mm", "phys_y_mm", "phys_z_mm",
    "bone_voxels", "bone_ratio", "seg_runtime_s", "ts_fast", "device",
]
display(df_meta[display_cols])

print(f"\n{'='*60}")
print("SEGMENTATION SUMMARY")
print(f"{'='*60}")
print(f"  Subjects processed: {df_meta['subject_id'].nunique()}")
print(f"  Knee volumes:       {len(df_meta)}")
print(f"  Bone ratio range:   {df_meta['bone_ratio'].min():.3f} – {df_meta['bone_ratio'].max():.3f}")
print(f"  Avg seg runtime:    {df_meta['seg_runtime_s'].mean():.0f}s per subject")
print(f"  Method:             TotalSegmentator (fast={TS_FAST}, device={DEVICE})")
print(f"  Boundary check:     {'ALL PASS' if all_pass else 'SOME FAIL'}")
print(f"\nOutputs:")
print(f"  Volumes: {OUTPUT_DIR}")
print(f"  Masks:   {MASK_DIR}")
print(f"  Meta:    {meta_path}")